In [1]:
"""
周期状态转移模型演示
"""

import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from models.periodic_model import PeriodicStateTransitionModel
from models.base_model import EventType

In [2]:
def demo_periodic_model():
    """演示周期状态转移模型 - T=2, n=4"""
    print("=" * 80)
    print("周期状态转移模型演示 (T=2, n=4)")
    print("=" * 80)
    
    # 创建模型实例 - T=2, n=4
    model = PeriodicStateTransitionModel(T=2, n=4)
    
    # 获取参数值
    T_val = model.model_params['T']  # T=2
    n_val = model.model_params['n']  # n=4
    
    print(f"\n📊 模型信息:")
    print(f"   参数: T={T_val}, n={n_val}")
    print(f"   初始状态: (0,1,0) 不健康")
    print(f"   周期长度: {2*T_val} 步 (不健康{T_val}步 → 健康{T_val}步 → 循环)")
    print(f"   完全转移条件: 连续 {n_val} 次干预")
    print(f"   可用干预: (0,1,0) 🔵 干预, (0,0,0) ⚪ 无干预")
    
    # 场景1: 无干预的自然周期性
    print("\n" + "=" * 80)
    print("场景1: 无干预 - 自然周期性交替 (T=2)")
    print("=" * 80)
    
    interventions_scene1 = [(0, 0, 0)] * 12
    
    result1 = model.simulate(interventions_scene1, return_details=True)
    _print_results(result1, model, title=f"自然周期 (T={T_val}: {T_val}步不健康 → {T_val}步健康 → 循环)")
    
    # 场景2: 部分干预（n=4 > T=2）
    print("\n" + "=" * 80)
    print(f"场景2: 部分干预 - 中断周期性但未达到完全转移 (n={n_val} > T={T_val})")
    print("=" * 80)
    
    # 在健康阶段进行3次干预（不足4次），中断了周期性但未完全转移
    interventions_scene2 = [
        (0, 0, 0),  # t=1: 无干预（不健康阶段第1步）
        (0, 0, 0),  # t=2: 无干预（不健康阶段第2步，即将转向健康）
        (0, 1, 0),  # t=3: 干预（健康阶段第1步）
        (0, 1, 0),  # t=4: 干预（健康阶段第2步，应该转向不健康但被干预打断）
        (0, 1, 0),  # t=5: 干预（继续干预）
        (0, 0, 0),  # t=6: 无干预（干预停止）
        (0, 0, 0),  # t=7: 无干预
        (0, 0, 0),  # t=8: 无干预
    ]
    
    result2 = model.simulate(interventions_scene2, return_details=True)
    _print_results(result2, model, title=f"健康阶段进行{n_val-1}次干预，中断周期性但未完全转移")
    
    # 场景3: 完全转移
    print("\n" + "=" * 80)
    print(f"场景3: 完全转移 - 连续 {n_val} 次干预")
    print("=" * 80)
    
    interventions_scene3 = [
        (0, 0, 0),  # t=1: 无干预
        (0, 0, 0),  # t=2: 无干预
    ] + [(0, 1, 0)] * n_val + [(0, 0, 0)] * 3
    
    result3 = model.simulate(interventions_scene3, return_details=True)
    _print_results(result3, model, title=f"连续{n_val}次干预 → 稳定健康")


def demo_periodic_visualization():
    """可视化周期性变化 (T=2, n=4)"""
    print("\n" + "=" * 80)
    print("周期性变化可视化 (T=2, n=4)")
    print("=" * 80)
    
    model = PeriodicStateTransitionModel(T=2, n=4)
    T_val = model.model_params['T']
    
    # 模拟16步无干预，观察周期性
    interventions = [(0, 0, 0)] * 16
    result = model.simulate(interventions)

    states = result["states"]
    
    print(f"\n状态序列（🟢=健康, 🔴=不健康，周期T={T_val}）:")
    print("-" * 60)
    
    for i, state in enumerate(states):
        if state == model.HEALTHY:
            symbol = "🟢"
        else:
            symbol = "🔴"
        print(f"{symbol}", end=" ")
        
        if (i + 1) % T_val == 0:
            print(f"  {i+1:2d}步", end="\n" if (i + 1) % (2 * T_val) == 0 else "  ")
    
    print(f"\n\n周期规律: " + "🔴"*T_val + "🟢"*T_val + "🔴"*T_val + "🟢"*T_val + "...")
    print(f"\n说明: 不健康{T_val}步 → 健康{T_val}步 → 循环")


def _print_results(result: dict, model, title: str = ""):
    """打印模拟结果"""
    if title:
        print(f"\n{title}")
    
    print(f"\n干预序列:")
    for t, inter in enumerate(result["interventions"], 1):
        if inter == model.INTERVENTION:
            print(f"   t={t:2d}: {inter} 🔵 干预")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 75)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<35}")
    print("-" * 75)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        state_name = "健康" if state == model.HEALTHY else "不健康"
        
        if i == 0:
            event_str = "-"        
        else:
            event = result["events"][i-1]
            if event == EventType.COMPLETE_TRANSITION:
                event_str = "E2a (完全转移) ⭐"
            elif event == EventType.PERIODIC_TO_UNHEALTHY:
                event_str = "E2b (周期性→不健康)"
            elif event == EventType.PERIODIC_TO_HEALTHY:
                event_str = "E2c (周期性→健康)"
            else:
                event_str = "E2d (状态维持)"        
        display_state = f"{state})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<35}")
    
    # 显示内部状态
    internal = model.get_internal_state()
    if internal["stable_healthy"]:
        print(f"\n💡 已进入稳定健康状态，周期性停止")


In [3]:
if __name__ == "__main__":
    demo_periodic_model()
    demo_periodic_visualization()

周期状态转移模型演示 (T=2, n=4)

📊 模型信息:
   参数: T=2, n=4
   初始状态: (0,1,0) 不健康
   周期长度: 4 步 (不健康2步 → 健康2步 → 循环)
   完全转移条件: 连续 4 次干预
   可用干预: (0,1,0) 🔵 干预, (0,0,0) ⚪ 无干预

场景1: 无干预 - 自然周期性交替 (T=2)

自然周期 (T=2: 2步不健康 → 2步健康 → 循环)

干预序列:
   t= 1: (0, 0, 0) ⚪ 无干预
   t= 2: (0, 0, 0) ⚪ 无干预
   t= 3: (0, 0, 0) ⚪ 无干预
   t= 4: (0, 0, 0) ⚪ 无干预
   t= 5: (0, 0, 0) ⚪ 无干预
   t= 6: (0, 0, 0) ⚪ 无干预
   t= 7: (0, 0, 0) ⚪ 无干预
   t= 8: (0, 0, 0) ⚪ 无干预
   t= 9: (0, 0, 0) ⚪ 无干预
   t=10: (0, 0, 0) ⚪ 无干预
   t=11: (0, 0, 0) ⚪ 无干预
   t=12: (0, 0, 0) ⚪ 无干预

状态演变:
---------------------------------------------------------------------------
步数     可观测状态                状态说明            事件                                 
---------------------------------------------------------------------------
0      (0, 1, 0))           不健康             -                                  
1      (0, 1, 0))           不健康             E2d (状态维持)                         
2      (0, 0, 0))           健康              E2c (周期性→健康)                       